# ViT-Light — Cross-Database Deepfake Detection Experiment

**Research question:** Does a Vision Transformer trained on one deepfake dataset generalize to others?

**Why ViT as the third model?**  
CNN-SRM captures local noise patterns; EfficientNetB0 captures local semantic texture.  
ViT uses **global self-attention across image patches** — a fundamentally different inductive bias  
with no convolution. It asks: *are there globally inconsistent relationships between distant regions?*

**Architecture (ViT-Tiny, pure Keras, no external dependencies):**
- Input: 224×224×3 → 196 patches of 16×16
- Patch embedding: Conv2D stride-16 → (B, 196, 192)
- Learnable positional embeddings
- 6× Transformer encoder blocks (LayerNorm → MHSA → Add → LayerNorm → MLP → Add)
- Global average pooling → Dense head → sigmoid
- ~5.7M parameters — safe for Tesla T4

**Training strategy:** Single-phase end-to-end training from random init (no pre-trained weights).  
Lower LR (1e-4) and more patience than EfficientNetB0 — ViT needs more epochs to converge.

In [ ]:
# =================== CELL 1: SETUP ===================
import os
import gc
import random
import pickle
import numpy as np
import tensorflow as tf
from datetime import datetime

os.environ['TF_XLA_FLAGS'] = '--tf_xla_enable_xla_devices=false'
os.environ['XLA_FLAGS'] = '--xla_gpu_cuda_data_dir=/usr/lib/nvidia-cuda-toolkit'
tf.config.optimizer.set_jit(False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print('✅ GPU memory growth enabled')
    except RuntimeError as e:
        print(e)

tf.keras.backend.clear_session()
print(f'✅ Setup complete — TensorFlow {tf.__version__}')

In [ ]:
# =================== CELL 2: PATHS & TENSORBOARD CONFIG ===================

GDRIVE_PATH   = os.path.expanduser('~/RealEyes/gdrive')
DATASET_ROOT  = os.path.join(GDRIVE_PATH, 'data_set_split')
DATASETS_DIR  = os.path.expanduser('~/RealEyes/RealEyes/datasets')
CELEBDF_DIR   = os.path.join(DATASETS_DIR, 'celebdf_v2')

MODEL_NAME            = 'vit'
EXPERIMENT_MODELS_DIR = '/home/sceuser/RealEyes/gdrive/deepfake_image_project/models/no_lora_experiment'  # no-LoRA run
EXPERIMENT_MODELS_DIR = '/home/sceuser/RealEyes/gdrive/deepfake_image_project/models/no_lora_experiment'  # no-LoRA run
os.makedirs(MODEL_DIR, exist_ok=True)

TB_LOG_ROOT = os.path.expanduser('~/RealEyes/tensorboard_logs')
os.makedirs(TB_LOG_ROOT, exist_ok=True)


def get_tb_log_dir(train_db_name, suffix='train'):
    ts = datetime.now().strftime('%Y%m%d_%H%M')
    return os.path.join(TB_LOG_ROOT, MODEL_NAME, f'{suffix}_{train_db_name}', ts)


def log_eval_to_tb(train_db_name, test_db_name, metrics: dict, step=0):
    log_dir = os.path.join(
        TB_LOG_ROOT, MODEL_NAME, 'cross_db_eval',
        f'train_{train_db_name}__test_{test_db_name}'
    )
    writer = tf.summary.create_file_writer(log_dir)
    with writer.as_default():
        for name, value in metrics.items():
            tf.summary.scalar(name, float(value), step=step)
    writer.flush()


print('✅ Paths ready')
print(f'  MODEL_DIR  : {MODEL_DIR}')
print(f'  TB_LOG_ROOT: {TB_LOG_ROOT}')
print()
print('▶  View TensorBoard:')
print(f'  [server ] tensorboard --logdir {TB_LOG_ROOT} --port 6006 --bind_all')
print( '  [local  ] ssh -L 6006:localhost:6006 <user>@<server_ip>')
print( '  [browser] http://localhost:6006')

In [ ]:
# =================== CELL 3: DATABASE DEFINITIONS ===================

DATABASES = {}


def _try_add(name, paths):
    missing = [k for k, v in paths.items() if not os.path.isdir(v)]
    if missing:
        print(f'  ⚠️  {name}: missing splits {missing} — skipped')
        return
    DATABASES[name] = paths
    print(f'  ✅ {name}')


print('🗂  Scanning databases...')

_try_add('OpenForensics', {
    'train': os.path.join(DATASETS_DIR, 'OpenForensicsV1/Dataset/Train'),
    'val':   os.path.join(DATASETS_DIR, 'OpenForensicsV1/Dataset/Validation'),
    'test':  os.path.join(DATASETS_DIR, 'OpenForensicsV1/Dataset/Test'),
})
_try_add('CustomWar', {
    'train': os.path.join(DATASET_ROOT, 'train'),
    'val':   os.path.join(DATASET_ROOT, 'val'),
    'test':  os.path.join(DATASET_ROOT, 'test'),
})
_try_add('CelebDF', {
    'train': os.path.join(CELEBDF_DIR, 'train'),
    'val':   os.path.join(CELEBDF_DIR, 'val'),
    'test':  os.path.join(CELEBDF_DIR, 'test'),
})
_try_add('CiFake', {
    'train': os.path.join(DATASETS_DIR, 'cifake/train'),
    'val':   os.path.join(DATASETS_DIR, 'cifake/test'),
    'test':  os.path.join(DATASETS_DIR, 'cifake/test'),
})

print(f'\n📋 Active databases: {list(DATABASES.keys())}')

In [ ]:
# =================== CELL 4: DATA LOADING HELPERS ===================

def load_dataset_images(dataset_path, max_images=None):
    valid_ext = {'.jpg', '.jpeg', '.png', '.gif', '.bmp'}
    image_paths, labels, skipped = [], [], 0

    for folder in sorted(os.listdir(dataset_path)):
        fpath = os.path.join(dataset_path, folder)
        if not os.path.isdir(fpath):
            continue
        fu = folder.upper()
        if fu == 'FAKE':
            label = 1
        elif fu == 'REAL':
            label = 0
        else:
            print(f'  ⚠️  Unknown folder "{folder}" — skipped')
            continue

        collected = []
        for root, _, files in os.walk(fpath):
            for fname in files:
                if os.path.splitext(fname)[1].lower() not in valid_ext:
                    skipped += 1
                    continue
                collected.append(os.path.join(root, fname))
        if max_images:
            collected = collected[:max_images]
        image_paths.extend(collected)
        labels.extend([label] * len(collected))

    if skipped:
        print(f'  ⚠️  {skipped} non-image files skipped')
    return np.array(image_paths), np.array(labels)


def load_db_split(db_name, split='train'):
    paths, labels = load_dataset_images(DATABASES[db_name][split])
    n_real = int(np.sum(labels == 0))
    n_fake = int(np.sum(labels == 1))
    print(f'    {db_name}/{split}: {len(paths):,} images  (REAL={n_real:,}, FAKE={n_fake:,})')
    return paths, labels


def compute_class_weights(labels):
    from sklearn.utils.class_weight import compute_class_weight
    classes = np.unique(labels)
    weights = compute_class_weight('balanced', classes=classes, y=labels)
    return {int(c): float(w) for c, w in zip(classes, weights)}


print('✅ Data loading helpers ready')

In [ ]:
# =================== CELL 5: VIT DATASET PIPELINE ===================
# ViT trains from scratch → normalize to [0, 1], no ImageNet preprocessing.
# Stronger augmentation to compensate for no pre-training.

IMG_SIZE = (224, 224)
AUTOTUNE = tf.data.AUTOTUNE


def _decode_normalize(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    return img, tf.cast(label, tf.float32)


def _augment(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.random_brightness(img, max_delta=0.15)
    img = tf.image.random_contrast(img, lower=0.80, upper=1.20)
    img = tf.image.random_saturation(img, lower=0.80, upper=1.20)
    img = tf.image.random_hue(img, max_delta=0.05)
    img = tf.clip_by_value(img, 0.0, 1.0)
    return img, label


def build_vit_dataset(image_paths, labels, batch_size=32, training=False):
    ds = tf.data.Dataset.from_tensor_slices((image_paths, labels))
    if training:
        ds = ds.shuffle(min(len(image_paths), 10000), reshuffle_each_iteration=True)
    ds = ds.map(_decode_normalize, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(_augment, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(4)
    options = tf.data.Options()
    options.autotune.ram_budget = 3 * 1024 * 1024 * 1024
    ds = ds.with_options(options)
    return ds


print('✅ ViT dataset pipeline ready')

In [ ]:
# =================== CELL 6: VIT-LIGHT MODEL BUILDER ===================
#
# Architecture: ViT-Tiny (pure Keras, no external dependencies)
#   - Patch embedding: Conv2D(192, 16, stride=16) → (B, 196, 192)
#   - Learnable 1D positional embedding
#   - 6× Transformer encoder blocks
#   - Global average pooling over patch sequence
#   - Dense(256, relu) → Dense(1, sigmoid)
#   - ~5.7M total parameters
#
# Compared to EfficientNetB0 (~5.3M) — similar size, different inductive bias:
#   EfficientNet: local convolution, translation equivariance, ImageNet priors
#   ViT-Light:    global self-attention, no spatial bias, random init

from tensorflow.keras import layers, Model

# ViT-Tiny hyper-parameters
VIT_IMAGE_SIZE  = 224
VIT_PATCH_SIZE  = 16
VIT_NUM_PATCHES = (VIT_IMAGE_SIZE // VIT_PATCH_SIZE) ** 2   # 196
VIT_HIDDEN_DIM  = 192
VIT_MLP_DIM     = 768    # 4 × hidden_dim
VIT_NUM_HEADS   = 3
VIT_NUM_LAYERS  = 6
VIT_DROPOUT     = 0.10


def build_vit_light():
    """Build ViT-Tiny deepfake detector from scratch."""
    inputs = layers.Input(shape=(VIT_IMAGE_SIZE, VIT_IMAGE_SIZE, 3), name='rgb_input')

    # ── Patch embedding ───────────────────────────────────────────────────────
    # Conv2D with kernel=patch_size and stride=patch_size extracts non-overlapping
    # patches and projects them to hidden_dim in a single operation.
    x = layers.Conv2D(
        filters=VIT_HIDDEN_DIM,
        kernel_size=VIT_PATCH_SIZE,
        strides=VIT_PATCH_SIZE,
        padding='valid',
        name='patch_embed'
    )(inputs)                                              # (B, 14, 14, 192)
    x = layers.Reshape(
        (VIT_NUM_PATCHES, VIT_HIDDEN_DIM),
        name='reshape_patches'
    )(x)                                                   # (B, 196, 192)

    # ── Positional embedding ──────────────────────────────────────────────────
    positions = tf.range(start=0, limit=VIT_NUM_PATCHES, delta=1)
    pos_emb = layers.Embedding(
        input_dim=VIT_NUM_PATCHES,
        output_dim=VIT_HIDDEN_DIM,
        name='pos_embed'
    )(positions)                                           # (196, 192)
    x = layers.Add(name='add_pos_embed')([x, pos_emb])
    x = layers.Dropout(VIT_DROPOUT, name='embed_dropout')(x)

    # ── Transformer encoder blocks ────────────────────────────────────────────
    for i in range(VIT_NUM_LAYERS):
        # --- Attention sub-block ---
        y = layers.LayerNormalization(epsilon=1e-6, name=f'ln1_{i}')(x)
        y = layers.MultiHeadAttention(
            num_heads=VIT_NUM_HEADS,
            key_dim=VIT_HIDDEN_DIM // VIT_NUM_HEADS,
            dropout=VIT_DROPOUT,
            name=f'mhsa_{i}'
        )(y, y)
        y = layers.Dropout(VIT_DROPOUT, name=f'attn_drop_{i}')(y)
        x = layers.Add(name=f'add_attn_{i}')([x, y])

        # --- MLP sub-block ---
        y = layers.LayerNormalization(epsilon=1e-6, name=f'ln2_{i}')(x)
        y = layers.Dense(VIT_MLP_DIM, name=f'mlp_fc1_{i}')(y)
        y = layers.Activation('gelu', name=f'mlp_gelu_{i}')(y)
        y = layers.Dropout(VIT_DROPOUT, name=f'mlp_drop1_{i}')(y)
        y = layers.Dense(VIT_HIDDEN_DIM, name=f'mlp_fc2_{i}')(y)
        y = layers.Dropout(VIT_DROPOUT, name=f'mlp_drop2_{i}')(y)
        x = layers.Add(name=f'add_mlp_{i}')([x, y])

    # ── Final layer norm + pooling ────────────────────────────────────────────
    x = layers.LayerNormalization(epsilon=1e-6, name='ln_final')(x)
    x = layers.GlobalAveragePooling1D(name='gap')(x)       # (B, 192)

    # ── Classification head ───────────────────────────────────────────────────
    x = layers.Dropout(0.30, name='head_drop1')(x)
    x = layers.Dense(256, activation='relu', name='head_dense1')(x)
    x = layers.Dropout(0.20, name='head_drop2')(x)
    outputs = layers.Dense(1, activation='sigmoid', name='prob_fake')(x)

    model = Model(inputs, outputs, name='ViT_Light_Deepfake')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.10),
        metrics=[
            'accuracy',
            tf.keras.metrics.AUC(name='auc'),
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall')
        ]
    )
    return model


print('✅ ViT-Light builder ready')
_tmp = build_vit_light()
print(f'  Total parameters: {_tmp.count_params():,}')
print(f'  Architecture: {VIT_NUM_LAYERS} blocks × ({VIT_NUM_HEADS} heads, dim={VIT_HIDDEN_DIM}, mlp={VIT_MLP_DIM})')
print(f'  Input patches: {VIT_NUM_PATCHES} × {VIT_PATCH_SIZE}px')
del _tmp
tf.keras.backend.clear_session()

In [ ]:
# =================== CELL 7: EVALUATION HELPERS ===================

from sklearn.metrics import classification_report, roc_auc_score, accuracy_score


def evaluate_model(model, test_ds):
    y_true, y_prob = [], []
    for x_batch, y_batch in test_ds:
        preds = model.predict(x_batch, verbose=0).flatten()
        y_true.extend(y_batch.numpy())
        y_prob.extend(preds)

    y_true = np.array(y_true).astype(int)
    y_prob = np.array(y_prob)
    y_pred = (y_prob > 0.5).astype(int)

    report = classification_report(
        y_true, y_pred, target_names=['REAL', 'FAKE'], output_dict=True, digits=4
    )
    metrics = {
        'accuracy':       float(accuracy_score(y_true, y_pred)),
        'roc_auc':        float(roc_auc_score(y_true, y_prob)),
        'f1_fake':        float(report['FAKE']['f1-score']),
        'f1_real':        float(report['REAL']['f1-score']),
        'precision_fake': float(report['FAKE']['precision']),
        'recall_fake':    float(report['FAKE']['recall']),
    }
    return metrics, report


def print_eval_report(train_db, test_db, metrics, report):
    tag = '  ✅ [WITHIN-DB]' if train_db == test_db else '  🔀 [CROSS-DB ]'
    print(f'\n{tag}  Trained on {train_db:<15} → Tested on {test_db}')
    sep = '  ' + '─' * 58
    print(sep)
    for cls in ['REAL', 'FAKE']:
        r = report[cls]
        print(f'  {cls:<6}  P={r["precision"]:.4f}  R={r["recall"]:.4f}  F1={r["f1-score"]:.4f}  n={int(r["support"]):,}')
    print(sep)
    print(f'  Accuracy={metrics["accuracy"]:.4f}   ROC-AUC={metrics["roc_auc"]:.4f}')


print('✅ Evaluation helpers ready')

In [ ]:
import ctypes
import traceback
# =================== CELL 8: MAIN EXPERIMENT LOOP ===================
#
# ViT trains from scratch — single training phase, no frozen/unfreeze stages.
# Higher patience (8 vs 5) because ViT converges slower than CNN-based models.
# LR: 1e-4 (lower than EfficientNet's 5e-4 for stable attention training)
# ─────────────────────────────────────────────────────────────────────────

BATCH_SIZE  = 32
all_results = {}

for train_db_name in DATABASES:
    print(f'\n{"="*70}')
    print(f'  🎯  TRAINING ViT-Light ON: {train_db_name}')
    print(f'{"="*70}')

    # ── 1. Load training data ─────────────────────────────────────────────
    print('\n📦 Loading training data...')
    train_paths, train_lbls = load_db_split(train_db_name, 'train')
    val_paths,   val_lbls   = load_db_split(train_db_name, 'val')
    class_weights           = compute_class_weights(train_lbls)
    print(f'  Class weights: {class_weights}')

    train_ds = build_vit_dataset(train_paths, train_lbls, batch_size=BATCH_SIZE, training=True)
    val_ds   = build_vit_dataset(val_paths,   val_lbls,   batch_size=BATCH_SIZE, training=False)

    # ── 2. Build fresh ViT ────────────────────────────────────────────────
    gc.collect()
    tf.keras.backend.clear_session()
    model     = build_vit_light()
    tb_log    = get_tb_log_dir(train_db_name, suffix='train')
    best_path = os.path.join(MODEL_DIR, f'trained_on_{train_db_name}.keras')

    # ── 3. Train end-to-end ───────────────────────────────────────────────
    print(f'\n🚀 Training ViT-Light end-to-end on {train_db_name}...')
    print(f'  Note: ViT trains from random init — needs ~20-30 epochs to converge.')
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_auc', mode='max', patience=8,
            restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_auc', mode='max', factor=0.5,
            patience=3, min_lr=1e-6, verbose=1),
        tf.keras.callbacks.ModelCheckpoint(
            best_path, monitor='val_auc', mode='max',
            save_best_only=True, verbose=1),
        tf.keras.callbacks.TensorBoard(
            log_dir=tb_log, histogram_freq=0, update_freq='epoch'),
    ]
    model.fit(
        train_ds, validation_data=val_ds,
        epochs=40, class_weight=class_weights,
        callbacks=callbacks, verbose=1
    )

    # ── 4. Evaluate on ALL databases ──────────────────────────────────────
    print(f'\n\n📊 EVALUATION — ViT-Light trained on {train_db_name}')
    best_model = tf.keras.models.load_model(best_path, compile=False)
    all_results[train_db_name] = {}

    for test_db_name in DATABASES:
        print(f'\n  🔍 Test database: {test_db_name}...')
        test_paths, test_lbls = load_db_split(test_db_name, 'test')
        test_ds               = build_vit_dataset(test_paths, test_lbls, batch_size=BATCH_SIZE, training=False)
        metrics, report       = evaluate_model(best_model, test_ds)
        all_results[train_db_name][test_db_name] = metrics
        log_eval_to_tb(train_db_name, test_db_name, metrics)
        print_eval_report(train_db_name, test_db_name, metrics, report)

    # ── 5. Memory cleanup ─────────────────────────────────────────────────
    del model, best_model, train_ds, val_ds
    gc.collect()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass
    tf.keras.backend.clear_session()
    print(f'\n✅ {train_db_name} experiment complete — GPU memory cleared.')

results_path = os.path.join(MODEL_DIR, 'all_results.pkl')
with open(results_path, 'wb') as f:
    pickle.dump(all_results, f)

print(f'\n\n{"="*70}')
print('  🏁  ALL EXPERIMENTS COMPLETE')
print(f'{"="*70}')
print(f'\nResults saved → {results_path}')

In [ ]:
# =================== CELL 9: CROSS-DATABASE RESULTS MATRIX ===================

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

db_names = list(DATABASES.keys())

for metric_key, metric_label, cmap in [
    ('roc_auc',  'ROC-AUC',  'YlOrRd'),
    ('accuracy', 'Accuracy', 'YlGn'),
    ('f1_fake',  'F1-FAKE',  'Blues'),
]:
    available = [d for d in db_names if d in all_results]
    matrix = [
        [all_results[tr].get(te, {}).get(metric_key, float('nan')) for te in db_names]
        for tr in available
    ]
    df = pd.DataFrame(
        matrix,
        index=[f'Train: {d}' for d in available],
        columns=[f'Test: {d}' for d in db_names]
    )

    fig, ax = plt.subplots(figsize=(9, 6))
    sns.heatmap(
        df.astype(float), annot=True, fmt='.3f',
        cmap=cmap, vmin=0.40, vmax=1.00, ax=ax,
        linewidths=0.5, linecolor='gray',
        cbar_kws={'label': metric_label}
    )
    ax.set_title(
        f'ViT-Light — Cross-Database {metric_label}\n'
        '(diagonal = within-database · off-diagonal = cross-database)',
        fontsize=13, fontweight='bold'
    )
    ax.set_ylabel('Training Database', fontsize=11)
    ax.set_xlabel('Test Database', fontsize=11)
    plt.xticks(rotation=30, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    fig_path = os.path.join(MODEL_DIR, f'cross_db_{metric_key}.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Figure saved: {fig_path}')
    print(f'\n{metric_label} Matrix:')
    print(df.to_string())
    print()

In [ ]:
# =================== CELL 10: TENSORBOARD LAUNCH INSTRUCTIONS ===================

print('━' * 60)
print('  📊  TENSORBOARD DASHBOARD')
print('━' * 60)
print()
print('1️⃣  On the SERVER terminal, run:')
print(f'   tensorboard --logdir {TB_LOG_ROOT} --port 6006 --bind_all')
print()
print('2️⃣  On your LOCAL machine, open an SSH tunnel:')
print('   ssh -L 6006:localhost:6006 <your_user>@<server_ip>')
print()
print('3️⃣  Open in browser:  http://localhost:6006')
print()
print('In TensorBoard SCALARS tab, filter by "vit" to see:')
print('  • Per-epoch loss / accuracy / auc for each training database')
print('  • ViT vs EfficientNet vs CNN-SRM convergence comparison')
print('    (all models log to the same TB_LOG_ROOT — toggle models in the UI)')
print()
print('Expected behavior for ViT:')
print('  • Larger databases (OpenForensics, CelebDF) → faster convergence')
print('  • Small databases (CustomWar ~7k) → slower convergence or higher val loss')
print('  • This gap vs EfficientNetB0 is itself a research finding.')